# Train code 

In [7]:
import os
import models._config as c
from dataset.LPIdataset import LPIDataset

data_dir = '/data/kiwan/LPI_KIWAN/STFT'
dataset = LPIDataset(data_dir, c.waveforms, data_type='pwnNoisy', model_type='UNet')

sample = dataset[0]
print(sample[0].shape, sample[1], sample[2])

torch.Size([3, 128, 128]) Barker -16


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import models._config as c

from tqdm import tqdm
from torch.utils.data import DataLoader
from models.LSTM import BiLSTM
from models.UNET import UNet, U2Net
from dataset.LPIdataset import LPIDataset

import torch.nn.utils.rnn as rnn_utils



def collate(batch):
    if len(batch[0])==4:  
        data, labels, lengths, snr = zip(*batch)
        data = [torch.tensor(seq, dtype=torch.float32) for seq in data]
        data_pad = rnn_utils.pad_sequence(data, batch_first=True)

        label_to_index = {label: idx for idx, label in enumerate(c.waveforms)}
        labels = torch.tensor([label_to_index[label] for label in labels], dtype=torch.long)
        lengths = torch.tensor(lengths, dtype=torch.int64)
        return data_pad, labels, lengths, snr

    else:  # UNet 또는 U2Net 모델
        data, labels, snr = zip(*batch)
        data = torch.stack(data, dim=0)
        label_to_index = {label: idx for idx, label in enumerate(c.waveforms)}
        labels = torch.tensor([label_to_index[label] for label in labels], dtype=torch.long)
        return data, labels, None, snr
    
def Train(model_type, data_dir, waveforms, datatype='pwnNoisy', batch_size=256, epochs=500, learning_rate=0.001, weight_decay=1e-5, device_ids=[0, 1]):
    CEloss = nn.CrossEntropyLoss()
    
    if model_type == 'BiLSTM':
        model = BiLSTM(input_size=2, hidden_size=128, num_layers=2, num_classes=len(waveforms))
    elif model_type == 'UNet':
        model = UNet(in_channels=3, out_channels=len(waveforms))
    elif model_type == 'U2Net':
        model = U2Net(in_channels=3, out_channels=len(waveforms))
    else:
        raise ValueError("Invalid model_type. Choose from 'BiLSTM', 'UNet', 'U2Net'.")
    
    model = nn.DataParallel(model, device_ids=device_ids).cuda()
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    dataset = LPIDataset(data_dir, waveforms, data_type=datatype, model_type=model_type)
    dataload = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)
    
    best_loss = float('inf')
    best_state = None
    losses = []

    for epoch in range(epochs):
        total_loss = 0.0
        progress = tqdm(enumerate(dataload), total=len(dataload), desc=f"Epoch {epoch + 1}/{epochs}", leave=False)

        for batch_idx, (data, labels, lengths, snrs) in progress:
            if model_type == 'BiLSTM':
                # BiLSTM은 시계열 입력
                data, labels = data.cuda(), labels.cuda()
                outputs = model(data, lengths)
            else:
                # UNet, U2Net은 TFI 이미지 입력
                data, labels = data.cuda(), labels.cuda()
                outputs = model(data)



            loss = CEloss(outputs, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            progress.set_postfix({'Batch': f'{batch_idx+1}/{len(dataload)}', 'Loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(dataload)
        losses.append(avg_loss)
        
        print(f'Epoch : {epoch+1}/{epochs}, Loss : {avg_loss:.4f}')
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = model.state_dict()
            torch.save(best_state, f'./ckpts/{datatype}/{model_type}_best_{best_loss:.4f}.pth')
            if best_loss < 0.002:
                break

        torch.cuda.empty_cache()
    
    torch.save(model.state_dict(), f'./ckpts/{datatype}/{model_type}_last_{best_loss:.4f}.pth')
    

model_type = 'UNet'
data_dir = '/data/kiwan/LPI_KIWAN/STFT'
datatypes = ['Signal', 'Noise', 'Noisy', 'pwnNoisy']
    
Train(
        model_type=model_type,
        batch_size=64,
        epochs=500,
        learning_rate=1e-3,
        weight_decay=1e-5,
        data_dir=data_dir,
        datatype=datatypes[-1],
        waveforms=c.waveforms
    )


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.rnn as rnn_utils
from tqdm import tqdm
from torch.utils.data import DataLoader

import models._config as c
from models.LSTM import BiLSTM
from dataset.LPIdataset import LPIDataset

    
datatypes = c.datatypes
       
def collate(batch):
    data, labels, lengths, _, _, _ = zip(*batch)
    data = [torch.tensor(seq, dtype=torch.float32) for seq in data]
    data_pad = rnn_utils.pad_sequence(data, batch_first=True)
    labels = torch.tensor([c.label_mapping[label] for label in labels], dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.int64)
    
    return data_pad, labels, lengths

torch.cuda.set_device(1)
CEloss = nn.CrossEntropyLoss()
model = BiLSTM(input_size=2, hidden_size=128, num_layers=2, num_classes=len(c.waveforms)).cuda()
# model = nn.DataParallel(model, device_ids=[0, 1]).cuda()
model.train()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

dataset = LPIDataset(c.data_dir, c.waveforms, data_type=datatypes[-1])              
dataload = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate)

best_loss = float('inf')
best_state = None

losses = []
epochs = 500
for epoch in range(epochs):
    total_loss = 0.0
    progress = tqdm(enumerate(dataload), total=len(dataload), desc=f"Epoch {epoch + 1}/{epochs}", leave=False)
        
    for batch_idx, (data, labels, lengths) in progress:
        optimizer.zero_grad()
        data, labels = data.cuda(), labels.cuda()
        outputs = model(data, lengths)
        loss = CEloss(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        progress.set_postfix({'Batch': f'{batch_idx+1}/{len(dataload)}', 'Loss': f'{loss.item():.4f}'})

        loss.detach()
    
    avg_loss = total_loss / len(dataload)
    losses.append(avg_loss)
    
    print(f'Epoch : {epoch+1}/{epochs}, Loss : {avg_loss:.4f}')
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = model.state_dict()
        torch.save(best_state, f'./models/{datatypes[-1]}_best_{best_loss}.pth')
        if best_loss < 0.01:
            break
    torch.cuda.empty_cache()
torch.save(model.state_dict(), f'./models/{datatypes[-1]}_last_{best_loss}.pth')


# dataset 전부 삭제하기 (데이터셋 테스트할 때 사용)

In [ ]:
import os
import shutil

data_dir = '/data/kiwan/LPI_KIWAN/'
files = os.listdir(data_dir)

for file in files:
    file_path = os.path.join(data_dir, file)
    if os.path.isfile(file_path):
        os.remove(file_path)
    elif os.path.isdir(file_path):
        shutil.rmtree(file_path)

print(f"All files in {data_dir} have been deleted.")


# 데이터셋 각 클래스 별 length 분포 확인

In [ ]:
import os
import numpy as np

class LPIDataset:
    def __init__(self, data_dir, waveforms, data_type='Signal'):
        self.data_dir = data_dir
        self.waveform = waveforms
        self.data_type = data_type
        self.file_list = self._collect()

    def _collect(self):
        files = []
        for waveform in self.waveform:
            waveform_folder = os.path.join(self.data_dir, waveform)
            files.extend([f for f in os.listdir(waveform_folder) if self.data_type in f])
        
        return files

    def _parse(self, file):
        parts = file.replace('.npy', '').split('_')
        label, snr, type, fps_idx = parts[0], int(parts[1].replace('snr', '')), parts[2], int(parts[3])
        
        return type, label, snr, fps_idx

    def _convIQ(self, complex_data):
        return complex_data.real, complex_data.imag
    
    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file = self.file_list[idx]
        type, label, snr, fps_idx= self._parse(file)
        file_path = os.path.join(self.data_dir, label, file)
        
        IQ_data = [self._convIQ(c) for c in np.load(file_path)]
        
        return IQ_data, label, len(IQ_data), snr, type, fps_idx 
    

data_dir = '/data/kiwan/LPI_KIWAN/'
datatypes = ['Signal', 'Noise', 'Noisy', 'pwnNoisy']
waveforms = ['LFM', 'Costas', 'Barker', 'Frank', 'P1', 'P2', 'P3', 'P4', 'T1', 'T2', 'T3', 'T4']
dataset = LPIDataset(data_dir, waveforms, data_type=datatypes[-1])


import matplotlib.pyplot as plt

class_lengths = {waveform: [] for waveform in waveforms}

for idx in range(len(dataset)):
    _, label, length, _, _, _ = dataset[idx]
    class_lengths[label].append(length)

fig, ax = plt.subplots(figsize=(12, 8))

for label, lengths in class_lengths.items():
    # if label in ['P3', 'P4']:
    #     count = sum(1 for length in lengths if length >= 2000)
    #     print(f'Number of {label} signals with length >= 2000: {count}')
    ax.hist(lengths, label=label, bins=50, alpha=0.6)

ax.set_title('Length Distribution per Class')
ax.set_xlabel('Length')
ax.set_ylabel('Frequency')
ax.legend()
plt.show()

In [ ]:
from models._config import C
from dataset.RadarDataset import RadarSignalDataset  


c = C()
if __name__ == "__main__":
    datajson = c.dataload(csv=True, mode='train')
    dataset = RadarSignalDataset(datajson, c.signalTypes[0:c.typeSize], snr_max=2)

In [ ]:
import matplotlib.pyplot as plt

class_lengths = {signal_type: [] for signal_type in set(dataset.labels)}

for idx in range(len(dataset)):
    _, label, _, length = dataset[idx]
    class_lengths[label].append(length)

fig, ax = plt.subplots(figsize=(12, 8))

for label, lengths in class_lengths.items():
    if label in ['T1', 'T2', 'T3', 'T4']:
        ax.hist(lengths, label=label, bins=50, alpha=0.6)

ax.set_title('Length Distribution per Class')
ax.set_xlabel('Length')
ax.set_ylabel('Frequency')
ax.legend()
plt.show()

In [ ]:
import numpy as np
import torch   
import torch.nn as nn
from torch.utils.data import DataLoader
from models.LSTM import BiLSTM
import matplotlib.pyplot as plt
from scipy.signal import stft

model = BiLSTM(input_size=2, hidden_size=128, num_layers=2, num_classes=12)
model.to(c.device)
model.eval()


fs = 100e6
batch_size = 4

# Filter dataset to include only specific labels
target_label = 'Costas'
target_indices = [idx for idx, (_, label, _, _) in enumerate(dataset) if label in target_label]
target_dataset = torch.utils.data.Subset(dataset, target_indices)
loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=False, collate_fn=model.collate)

def logscale(x, eb=1e-6):
    return np.log(np.abs(x)+eb)



for batch_data, batch_label, batch_snr, batch_length in loader:
    batch_data = batch_data.to(c.device)
    fig, axes = plt.subplots(batch_size, 10, figsize=(30, 15))
    for i, sample in enumerate(zip(batch_data, batch_label, batch_snr, batch_length)):
        data, label, _, length = sample
        label = c.signalTypes[label]
        
        
        x_I = data[:length, 0].cpu().numpy()
        x_Q = data[:length, 1].cpu().numpy()
        
        f, t, Zxx = stft(x_I + x_Q, fs=fs, nperseg=256)
        f_I, t_I, Zxx_I = stft(x_I, fs=fs, nperseg=256)
        f_Q, t_Q, Zxx_Q = stft(x_Q, fs=fs, nperseg=256)
        
        logx_I = logscale(x_I, eb=1e-3)
        logx_Q = logscale(x_Q, eb=1e-3)
        
        log_f, log_t, log_Zxx = stft(logx_I + logx_Q, fs=fs, nperseg=256)
        log_f_I, log_t_I, log_Zxx_I = stft(logx_I, fs=fs, nperseg=256)
        log_f_Q, log_t_Q, log_Zxx_Q = stft(logx_Q, fs=fs, nperseg=256)
        
        axes[i, 0].plot(x_I, label='I')
        axes[i, 1].plot(x_Q, label='Q')
        axes[i, 2].plot(logx_I, label='I')
        axes[i, 3].plot(logx_Q, label='Q')
        axes[i, 4].pcolormesh(t_I, f_I, np.abs(Zxx_I), shading='gouraud')
        axes[i, 5].pcolormesh(t_Q, f_Q, np.abs(Zxx_Q), shading='gouraud')
        axes[i, 6].pcolormesh(log_t_I, log_f_I, np.abs(log_Zxx_I), shading='gouraud')
        axes[i, 7].pcolormesh(log_t_Q, log_f_Q, np.abs(log_Zxx_Q), shading='gouraud')
        axes[i, 8].pcolormesh(t, f, np.abs(Zxx), shading='gouraud')
        axes[i, 9].pcolormesh(log_t, log_f, np.abs(log_Zxx), shading='gouraud')
        
        
        axes[i, 0].set_title(f'I-{label}')
        axes[i, 1].set_title(f'Q-{label}')
        axes[i, 2].set_title(f'logscaled-I-{label}')
        axes[i, 3].set_title(f'logscaled-Q-{label}')
        axes[i, 4].set_title(f'STFT-{label}')
        axes[i, 5].set_title(f'logscaled-STFT-{label}')
        
    break
    # with torch.no_grad():
    #     outputs = model(batch_data, batch_length)
        
    #     break
    
    plt.tight_layout()
    

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Define the directory containing the dataset
dataset_directory = '/data/kiwan/LPI_Radar_1000_'
sclass = ['Barker', 'Costas', 'Frank', 'LFM', 'P1', 'P2', 'P3', 'P4', 'T1', 'T2', 'T3', 'T4']
j = 0
for cls in sclass:
    print(cls, end=' ')
    pth = os.path.join(dataset_directory, cls)
    files = os.listdir(pth)
    i = 0
    for file in files:
        if not file.endswith('-noisy.npy'):
            continue
        # print('Processing', cls, file)
        data = np.load(os.path.join(pth, file))
        print(data.shape, end=' ')
        data = data[:, 0]
        # plt.plot(data.shape)
        # plt.show()
        i = i + 1
        if i == 5:
            break
    print()
    j = j + 1
    if j == 12:
        break   
